# When experiments go wrong: practice notebook

**Causal Analytics · Asynchronous module** · Simon Business School · Prof. Tianli Xia

The code from the module's web page, the simulations behind its slides, and short exercises (cells marked `# YOUR CODE HERE`), with regressions in `statsmodels`.

**In Colab:** *File → Save a copy in Drive* first. Run cells with **Shift+Enter**.

## 0. Setup

The résumé-audit data (Bertrand & Mullainathan 2004): one row per résumé, `afam` = 1 if the name sounds Black, `call` = 1 if the employer called back. The cell reads a local copy, or the copy on the module's website, or rebuilds the two columns from the published counts.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from itertools import combinations
from pathlib import Path

SITE = "https://tianlixia.com/causal-analytics/A3/"   # the module's website, e.g. "https://<user>.github.io/<repo>/A3/" (filled in when the site is published)

local = [Path(p) for p in ["resume_calls.csv", "data/resume_calls.csv", "web/data/resume_calls.csv"]]
found = [p for p in local if p.exists()]
resume = None
if found:
    resume = pd.read_csv(found[0])
elif SITE:
    try:
        resume = pd.read_csv(SITE + "data/resume_calls.csv")
    except Exception as err:
        print("website copy not reachable:", err)
if resume is None:   # 157 of 2,435 Black-sounding and 235 of 2,435 white-sounding names got a callback
    resume = pd.DataFrame({"afam": np.r_[np.ones(2435), np.zeros(2435)].astype(int),
                           "call": np.r_[np.ones(157), np.zeros(2278), np.ones(235), np.zeros(2200)].astype(int)})
resume.groupby("afam")["call"].agg(["mean", "count"])

## Part A. How many "significant" imbalances are normal?

One perfectly randomized experiment: 2,000 customers, a coin flip each, ten pre-treatment variables that the coin cannot have affected. Then a $t$-test of each variable on treatment.

In [ ]:
rng = np.random.default_rng(3)
n, k = 2_000, 10
X = pd.DataFrame(rng.normal(size=(n, k)), columns=[f"x{j + 1}" for j in range(k)])
X["d"] = rng.integers(0, 2, n)
pvals = {c: smf.ols(f"{c} ~ d", data=X).fit().pvalues["d"] for c in X.columns[:k]}
print(pd.Series(pvals).round(3).to_string())
joint = smf.ols("d ~ " + " + ".join(X.columns[:k]), data=X).fit()
print(f"joint F-test p-value: {joint.f_pvalue:.3f}")

**Exercise A.** Repeat this 1,000 times (you can skip the regressions: with random assignment each variable's $t$-statistic is approximately standard normal, so draw `rng.normal(size=(1000, 10))` and count $|t| > 1.96$). What share of experiments has at least one significant row? Compare with $1 - 0.95^{10}$.

In [ ]:
# YOUR CODE HERE

## Part B. Two schools with different treatment shares

School A: 1,500 students, 80% treated, effect 7, baseline 60. School B: 500 students, 30% treated, effect 3, baseline 40. The ATE is 6.

In [ ]:
rng = np.random.default_rng(7)
nA, nB, pA, pB = 1500, 500, 0.8, 0.3
school = np.r_[np.zeros(nA), np.ones(nB)].astype(int)            # 0 = A, 1 = B
y0 = np.where(school == 0, 60, 40) + rng.normal(0, 8, nA + nB)
tau = np.where(school == 0, 7.0, 3.0)
d = np.where(school == 0, rng.random(nA + nB) < pA, rng.random(nA + nB) < pB).astype(int)
df = pd.DataFrame({"y": y0 + tau * d, "d": d, "school": school})

naive = df.y[df.d == 1].mean() - df.y[df.d == 0].mean()
dummy = smf.ols("y ~ d + school", data=df).fit().params["d"]
within = {s: g.y[g.d == 1].mean() - g.y[g.d == 0].mean() for s, g in df.groupby("school")}
weighted = (nA * within[0] + nB * within[1]) / (nA + nB)
print(f"ATE {tau.mean():.2f} | pooled {naive:.2f} | school dummy {dummy:.2f} | within, weighted by size {weighted:.2f}")

**Exercise B.** Set both schools to treat 50% (`pA = pB = 0.5`) and rerun. What happens to the pooled difference, and why?

In [ ]:
# YOUR CODE HERE

## Part C. Non-compliance: five estimators

Types are drawn at random (50% compliers, 30% never-takers, 20% always-takers); never-takers have low and always-takers high outcomes without treatment. The true effect on compliers is 4.

In [ ]:
rng = np.random.default_rng(2024)
N = 20_000
kind = rng.choice(["c", "n", "a"], size=N, p=[0.5, 0.3, 0.2])   # complier / never-taker / always-taker
z = rng.integers(0, 2, N)                                        # random assignment
d = np.where(kind == "c", z, np.where(kind == "a", 1, 0))         # treatment received
y0 = np.select([kind == "c", kind == "n", kind == "a"], [50, 40, 65]) + rng.normal(0, 10, N)
y1 = y0 + np.select([kind == "c", kind == "n", kind == "a"], [4, 2, 6])
y = d * y1 + (1 - d) * y0

itt = y[z == 1].mean() - y[z == 0].mean()                        # the effect of the offer
first_stage = d[z == 1].mean() - d[z == 0].mean()                # share of compliers
pp = ((z == 1) & (d == 1)) | ((z == 0) & (d == 0))
print(f"ITT           {itt:6.2f}")
print(f"first stage   {first_stage:6.3f}")
print(f"Wald (LATE)   {itt / first_stage:6.2f}   (true effect on compliers: 4)")
print(f"as treated    {y[d == 1].mean() - y[d == 0].mean():6.2f}")
print(f"per protocol  {y[pp & (z == 1)].mean() - y[pp & (z == 0)].mean():6.2f}")

The same Wald ratio from two regressions (the live session on instrumental variables runs it as two-stage least squares, which also gives the standard error):

In [ ]:
nc = pd.DataFrame({"y": y, "z": z, "d": d})
itt_reg = smf.ols("y ~ z", data=nc).fit()
fs_reg = smf.ols("d ~ z", data=nc).fit()
print(f"ITT {itt_reg.params['z']:.2f} (SE {itt_reg.bse['z']:.2f}) / first stage {fs_reg.params['z']:.3f} = {itt_reg.params['z'] / fs_reg.params['z']:.2f}")
print(f"approximate SE of the ratio: {itt_reg.bse['z'] / fs_reg.params['z']:.2f}")

**Exercise C.** Make non-compliance one-sided: no always-takers (shares `[0.6, 0.4, 0.0]`). Compute the ITT, the compliance rate, their ratio, and the true average effect among the treated. Do they match?

In [ ]:
# YOUR CODE HERE

## Part D. Randomization inference: shuffle the labels

In [ ]:
# eight units: all 70 ways to pick the four treated
Y = np.array([12, 4, 6, 10, 6, 0, 1, 1.])
actual = Y[:4].mean() - Y[4:].mean()                       # 6
diffs = []
for treated in combinations(range(8), 4):
    t = np.zeros(8, bool); t[list(treated)] = True
    diffs.append(Y[t].mean() - Y[~t].mean())
diffs = np.array(diffs)
print(f"eight units: {np.sum(np.abs(diffs) >= actual)} of 70 shuffles as large -> p = {np.mean(np.abs(diffs) >= actual):.3f}")

# the resume audit: shuffle the names 20,000 times
black, call = resume["afam"].to_numpy().astype(bool), resume["call"].to_numpy()
gap = call[black].mean() - call[~black].mean()
rng = np.random.default_rng(1)
shuffled = np.array([(lambda b: call[b].mean() - call[~b].mean())(rng.permutation(black)) for _ in range(20_000)])
print(f"resume audit: gap {100 * gap:.2f} points; shuffles at least as large: {np.sum(np.abs(shuffled) >= abs(gap))} of 20,000")
print(f"spread of the shuffled gaps: {100 * shuffled.std():.2f} points")

plt.hist(100 * shuffled, bins=50, color="#9aa3b8")
plt.axvline(100 * gap, color="#ea580c", lw=2)
plt.xlabel("gap after shuffling the names (percentage points)")
plt.show()

The regression gives the same verdict, and its standard error matches the spread of the shuffled gaps:

In [ ]:
m = smf.ols("call ~ afam", data=resume).fit()
print(f"OLS: gap {100 * m.params['afam']:.2f} points, SE {100 * m.bse['afam']:.2f}, p = {m.pvalues['afam']:.5f}")

**Exercise D.** (1) For the eight units, test the *median* difference instead of the mean: what share of the 70 assignments gives a median difference at least as large (in absolute value) as the actual one? (2) Take a random 200 résumés (`resume.sample(200, random_state=4)`) and compare the permutation $p$-value with the OLS $p$-value.

In [ ]:
# YOUR CODE HERE